In [ ]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.neighbors import KNeighborsRegressor

# 1. CONECTAR FUNCIONES
ruta_hugo = Path("../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

from channel_estimator import compute_channel_matrix_from_iq_paths
from channel_features_complete import extract_channel_matrix_features

RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("Datos TEST Vaso carton")

# Seleccionamos TODAS las variables numéricas que saque Hugo
print("📡 Extrayendo firmas radar del Vaso de Cartón...")
datos_train = []
datos_test = []

carpetas = {"frio": 15.0, "templado": 40.0, "caliente": 90.0}

for carpeta, temp in carpetas.items():
    ruta = BBDD_DIR / carpeta
    if not ruta.exists(): continue
    
    for rx_path in ruta.glob("iq_rx_*.bin"):
        tx_path = ruta / rx_path.name.replace("rx", "tx")
        
        if tx_path.exists():
            # Extraer características
            H = compute_channel_matrix_from_iq_paths(
                tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                output_order="mk", verbose=False
            )
            features = extract_channel_matrix_features(H, input_order="mk")
            features["temperatura"] = temp
            
            # Separar: Muestra 1 para estudiar, Muestra 2 para el examen
            if "1.bin" in rx_path.name:
                datos_train.append(features)
            else:
                datos_test.append(features)

# 2. ENTRENAMIENTO DEL MODELO
df_train = pd.DataFrame(datos_train).dropna(axis=1)
df_test = pd.DataFrame(datos_test)[df_train.columns] # Asegurar mismas columnas

X_train = df_train.drop(columns=["temperatura"])
y_train = df_train["temperatura"]
X_test = df_test.drop(columns=["temperatura"])
y_test = df_test["temperatura"]

# Usamos KNN: Al tener pocas muestras y mucho ruido, memorizar a los "vecinos" es más seguro que un Random Forest
modelo_rescate = KNeighborsRegressor(n_neighbors=1) 
modelo_rescate.fit(X_train, y_train)

# 3. RESULTADOS DE LA DEMO
print("\n" + "="*50)
print("🎯 RESULTADOS DEL MODELO DE RESCATE (Testando con Muestra 2)")
print("="*50)

predicciones = modelo_rescate.predict(X_test)
for real, pred in zip(y_test, predicciones):
    print(f"   Vaso Real a {real} ºC -> El Radar predice: {pred:.1f} ºC")
print("="*50)

# (Opcional) Guardarlo para usarlo el día de la presentación
import joblib
joblib.dump(modelo_rescate, 'modelo_demo.pkl')
joblib.dump(list(X_train.columns), 'variables_demo.pkl')